In [3]:
import hashlib
import pandas as pd

def _generate_hash(row):
    values = []

    for v in row:
        if pd.isna(v):
            v = ""
        v = str(v).strip().lower()
        values.append(v)

    row_str = "|".join(values)
    return hashlib.md5(row_str.encode()).hexdigest()


def write_df_safe(writer, df, sheet_name, note_if_empty=None):
    if isinstance(df, pd.DataFrame) and not df.empty:
        df.to_excel(writer, sheet_name=sheet_name, index=False)
    else:
        placeholder = pd.DataFrame({
            "Info": [note_if_empty or "No data available"]
        })
        placeholder.to_excel(writer, sheet_name=sheet_name, index=False)


path = r"C:\Users\kaustubh.keny\Downloads\ibbi_orders.xlsx"
sheets_dict = pd.read_excel(path, sheet_name=None)

new_data = {}

for sheetName, df in sheets_dict.items():
    df = df.copy()
    df["hash_id"] = df.apply(_generate_hash, axis=1)
    new_data[sheetName] = df
    
    new_data[sheetName] = df
    
with pd.ExcelWriter(path, engine="openpyxl") as writer:
    for name, df in new_data.items():
        write_df_safe(writer, df, name[:31])
    

In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import time
import os

DOWNLOAD_DIR = os.path.abspath("downloads")

chrome_options = webdriver.ChromeOptions()
chrome_options.add_experimental_option(
    "prefs",
    {
        "download.default_directory": DOWNLOAD_DIR,
        "download.prompt_for_download": False,
        "directory_upgrade": True,
        "safebrowsing.enabled": True,
    },
)


driver = webdriver.Chrome(
    # service=Service(ChromeDriverManager().install()),
    options=chrome_options,
)

wait = WebDriverWait(driver, 30)
driver.get("https://tradestat.commerce.gov.in/meidb/country_wise_all_commodities_export")

# Month
Select(wait.until(EC.element_to_be_clickable((By.ID, "cwcexddMonth"))))\
    .select_by_value("1")  # January

# Year
Select(driver.find_element(By.ID, "cwcexddYear"))\
    .select_by_value("2026")

# Country (example: U S A = 423)
Select(driver.find_element(By.ID, "cwcexallcount"))\
    .select_by_value("423")

# Commodity Level (6 digit)
Select(driver.find_element(By.ID, "cwcexddCommodityLevel"))\
    .select_by_value("6")

# Values in (US $ Million)
Select(driver.find_element(By.ID, "cwcexddReportVal"))\
    .select_by_value("1")

# Year Type (Financial Year)
Select(driver.find_element(By.ID, "cwcexddReportYear"))\
    .select_by_value("1")

driver.find_element(By.XPATH, "//button[@type='submit']").click()

# Wait until Excel button appears
excel_btn = wait.until(
    EC.element_to_be_clickable(
        (By.CSS_SELECTOR, "button.buttons-excel.buttons-html5")
    )
)

excel_btn.click()

# Wait for file to download
time.sleep(10)
print("✅ Excel download triggered")

driver.quit()

✅ Excel download triggered
